# **Teacher–Student distillation**

Hybrid Feature Loss (L2 + InfoNCE)

KL Logit Distillation

Gradignan++ attention alignment

Softplus temperature stabilization

Metrics: Accuracy, Precision, Recall, F1

Model checkpoint saving

**Teacher:** ResNet‑50

**Student:** MobileNetV2

**Expriment Dataset:** CIFAR10

## Full PyTorch Script

In [ ]:
## Full PyTorch Script
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

device = "cuda" if torch.cuda.is_available() else "cpu"

## Dataset (CIFAR-100)

In [ ]:

transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

trainset = torchvision.datasets.CIFAR100(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

testset = torchvision.datasets.CIFAR100(
    root='./data',
    train=False,
    download=True,
    transform=transforms.ToTensor()
)

trainloader = torch.utils.data.DataLoader(
    trainset,
    batch_size=128,
    shuffle=True
)

testloader = torch.utils.data.DataLoader(
    testset,
    batch_size=128,
    shuffle=False
)

100%|██████████| 170M/170M [00:35<00:00, 4.77MB/s]


## Teacher and Student Networks

In [ ]:
teacher = torchvision.models.resnet50(pretrained=True)
teacher.fc = nn.Linear(2048, 10)

# Create a feature extractor for the teacher model (all layers except the final fc layer)
teacher_feature_extractor = nn.Sequential(*list(teacher.children())[:-1])

student = torchvision.models.mobilenet_v2(pretrained=False)
student.classifier[1] = nn.Linear(1280, 10)

# Add a projection layer to the student to match teacher's feature dimension
student.feature_projector = nn.Linear(1280, 2048)

teacher = teacher.to(device)
student = student.to(device)

teacher.eval()

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 126MB/s]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

##Hybrid Feature Loss (L2 + InfoNCE)

In [ ]:

class HybridFeatureLoss(nn.Module):

    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = nn.Parameter(torch.tensor(temperature))

    def forward(self, s_feat, t_feat):

        l2 = F.mse_loss(s_feat, t_feat)

        s = F.normalize(s_feat, dim=1)
        t = F.normalize(t_feat, dim=1)

        logits = torch.mm(s, t.T)

        temp = F.softplus(self.temperature)

        logits = logits / temp

        labels = torch.arange(logits.size(0)).to(logits.device)

        infonce = F.cross_entropy(logits, labels)

        return l2 + infonce

KL Distillation Loss

In [ ]:

class LogitKLLoss(nn.Module):

    def __init__(self, T=4):
        super().__init__()
        self.T = T

    def forward(self, s_logits, t_logits):

        p_s = F.log_softmax(s_logits / self.T, dim=1)
        p_t = F.softmax(t_logits / self.T, dim=1)

        loss = F.kl_div(p_s, p_t, reduction="batchmean") * (self.T**2)

        return loss

## Grad-CAM++ Attention

In [ ]:

def gradcampp(feature, gradient):

    weights = gradient.pow(2) / (
        2*gradient.pow(2) +
        torch.sum(feature * gradient.pow(3),
        dim=(2,3), keepdim=True) + 1e-6
    )

    weights = torch.sum(weights * F.relu(gradient),
                        dim=(2,3), keepdim=True)

    cam = F.relu(torch.sum(weights * feature, dim=1, keepdim=True))

    cam = F.interpolate(cam, size=(32,32), mode='bilinear')

    cam = cam / (cam.max() + 1e-6)

    return cam

## Improved DisWOT Loss

In [ ]:

class ImprovedDisWOTLoss(nn.Module):

    def __init__(self):
        super().__init__()

        self.feature_loss = HybridFeatureLoss()
        self.kl_loss = LogitKLLoss()

        self.lambda_feat = 1.0
        self.lambda_kl = 1.0
        self.lambda_cam = 1.0

    def forward(
        self,
        s_feat,
        t_feat,
        s_logits,
        t_logits,
        s_cam,
        t_cam
    ):

        L_feat = self.feature_loss(s_feat, t_feat)

        L_kl = self.kl_loss(s_logits, t_logits)

        L_cam = F.mse_loss(s_cam, t_cam)

        total = (
            self.lambda_feat * L_feat +
            self.lambda_kl * L_kl +
            self.lambda_cam * L_cam
        )

        return total

## Metrics Function

In [ ]:

def compute_metrics(y_true, y_pred):

    acc = accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    return acc, precision, recall, f1

## Training Loop

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re


criterion = ImprovedDisWOTLoss()

optimizer = torch.optim.Adam(student.parameters(), lr=1e-3)

for epoch in range(200):

    student.train()

    all_preds = []
    all_labels = []

    for images, labels in trainloader:

        images = images.to(device)
        labels = labels.to(device)

        with torch.no_grad():
            t_logits = teacher(images)
            # Extract features using the defined feature extractor
            t_feat_raw = teacher_feature_extractor(images)
            t_feat = t_feat_raw.view(t_feat_raw.size(0), -1) # Flatten features

        s_logits = student(images)
        s_feat_raw = student.features(images).mean([2,3])
        # Project student features to match teacher's dimension
        s_feat = student.feature_projector(s_feat_raw)

        s_feat.requires_grad_(True)

        s_logits.sum().backward(retain_graph=True)

        grad = s_feat.grad if s_feat.grad is not None else torch.zeros_like(s_feat)

        t_cam = torch.zeros(images.size(0),1,32,32).to(device)
        s_cam = torch.zeros(images.size(0),1,32,32).to(device)

        loss = criterion(
            s_feat,
            t_feat,
            s_logits,
            t_logits,
            s_cam,
            t_cam
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        preds = torch.argmax(s_logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    acc, precision, recall, f1 = compute_metrics(
        np.array(all_labels),
        np.array(all_preds)
    )

    print(
        f"Epoch {epoch+1} | "
        f"Loss {loss.item():.3f} | "
        f"Acc {acc:.3f} | "
        f"Prec {precision:.3f} | "
        f"Recall {recall:.3f} | "
        f"F1 {f1:.3f}"
    )

    torch.save(
        student.state_dict(),
        "student_diswot_improved.pth"
    )


    metrics_data = []
for line in training_output.strip().split('\n'):
    match = re.match(r'Epoch (\d+) \| Loss (\d+\.\d+) \| Acc (\d+\.\d+) \| Prec (\d+\.\d+) \| Recall (\d+\.\d+) \| F1 (\d+\.\d+)', line)
    if match:
        epoch, loss, acc, prec, recall, f1 = match.groups()
        metrics_data.append({
            'Epoch': int(epoch),
            'Loss': float(loss),
            'Accuracy': float(acc),
            'Precision': float(prec),
            'Recall': float(recall),
            'F1 Score': float(f1)
        })

metrics_df = pd.DataFrame(metrics_data)

if not metrics_df.empty:
    print("Metrics extracted successfully:")
    display(metrics_df.head())
else:
    print("No metrics found. Please ensure the training_output variable contains valid data.")


In [ ]:
if not metrics_df.empty:
    plt.figure(figsize=(10, 6))
    sns.lineplot(x='Epoch', y='Loss', data=metrics_df)
    plt.title('Training Loss per Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True)
    plt.show()
else:
    print("Cannot plot Loss: Metrics DataFrame is empty.")


In [ ]:
if not metrics_df.empty:
    plt.figure(figsize=(10, 6))
    sns.lineplot(x='Epoch', y='Accuracy', data=metrics_df)
    plt.title('Training Accuracy per Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.grid(True)
    plt.show()
else:
    print("Cannot plot Accuracy: Metrics DataFrame is empty.")


In [ ]:
if not metrics_df.empty:
    plt.figure(figsize=(10, 6))
    sns.lineplot(x='Epoch', y='F1 Score', data=metrics_df)
    plt.title('Training F1 Score per Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('F1 Score')
    plt.grid(True)
    plt.show()
else:
    print("Cannot plot F1 Score: Metrics DataFrame is empty.")


In [ ]:
class FeatureHook:
    def __init__(self):
        self.feature = None
        self.gradient = None

    def forward_hook(self, module, input, output):
        self.feature = output

    def backward_hook(self, module, grad_input, grad_output):
        # grad_output[0] contains the gradient of the output w.r.t the input of this layer
        self.gradient = grad_output[0]

In [ ]:
teacher_hook = FeatureHook()
student_hook = FeatureHook()

# Register hooks for Teacher (ResNet-50)
# The last convolutional layer in ResNet-50 is typically conv3 within the last Bottleneck block of layer4
teacher.layer4[-1].conv3.register_forward_hook(teacher_hook.forward_hook)
teacher.layer4[-1].conv3.register_full_backward_hook(teacher_hook.backward_hook)

# Register hooks for Student (MobileNetV2)
# The last convolutional layer in MobileNetV2 features before the classifier
# It's usually the last Conv2d within the last InvertedResidual block's conv sequential
student.features[-1].conv[3].register_forward_hook(student_hook.forward_hook)
student.features[-1].conv[3].register_full_backward_hook(student_hook.backward_hook)

print("Hooks registered successfully for teacher and student models.")

In [ ]:
criterion = ImprovedDisWOTLoss()

optimizer = torch.optim.Adam(student.parameters(), lr=1e-3)

for epoch in range(200):

    student.train()

    all_preds = []
    all_labels = []

    for images, labels in trainloader:

        images = images.to(device)
        labels = labels.to(device)

        # --- Teacher Pass for Features, Logits, and CAM gradients ---
        with torch.no_grad():
            t_logits = teacher(images)
            t_feat_raw = teacher_feature_extractor(images)
            t_feat = t_feat_raw.view(t_feat_raw.size(0), -1)

        # To get teacher CAM gradients, we need a separate forward pass
        # and then a backward pass with a dummy loss if not training teacher
        teacher.zero_grad()
        t_logits_for_grad = teacher(images)
        # Target one class for backprop (e.g., the predicted class or a specific class)
        # For simplicity, we can backpropagate on the sum of logits or the max logit
        # Here, we'll backpropagate the sum to get general gradients for CAM
        t_logits_for_grad[:, t_logits_for_grad.argmax(dim=1)].sum().backward(retain_graph=True)

        # --- Student Pass for Features, Logits, and CAM gradients ---
        student.zero_grad()
        s_logits = student(images)
        s_feat_raw = student.features(images)
        s_feat = student.feature_projector(s_feat_raw.mean([2,3])) # Project student features after pooling

        # Perform backward pass for student logits to get gradients for CAM
        s_logits[:, s_logits.argmax(dim=1)].sum().backward(retain_graph=True)

        # Get CAMs
        # For ResNet-50, teacher_hook.feature will be from layer4[-1].conv3, which is 256 channels
        # The `gradcampp` function expects a 4D feature map (N, C, H, W)
        t_cam = gradcampp(teacher_hook.feature, teacher_hook.gradient)

        # For MobileNetV2, student_hook.feature will be from features[-1].conv[3], which is 320 channels (if output_channels was 320 for that conv block)
        s_cam = gradcampp(student_hook.feature, student_hook.gradient)

        loss = criterion(
            s_feat,
            t_feat,
            s_logits,
            t_logits,
            s_cam,
            t_cam
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        preds = torch.argmax(s_logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    acc, precision, recall, f1 = compute_metrics(
        np.array(all_labels),
        np.array(all_preds)
    )

    print(
        f"Epoch {epoch+1} | "
        f"Loss {loss.item():.3f} | "
        f"Acc {acc:.3f} | "
        f"Prec {precision:.3f} | "
        f"Recall {recall:.3f} | "
        f"F1 {f1:.3f}"
    )

    torch.save(
        student.state_dict(),
        "student_diswot_improved.pth"
    )